# 🎵 CodeAlpha Internship Project 3: Music Generation with AI
**Author:** AI Intern | **Track:** Artificial Intelligence | **Target Platform:** Google Colab (Free GPU Tier)

---

### 📌 Project Overview
This notebook implements an end-to-end Deep Learning pipeline to generate original polyphonic piano music using **Long Short-Term Memory (LSTM)** neural networks and **Music21**.

### 🌟 Key Features:
1. **Curated Public-Domain Dataset:** Authentic classical MIDI pieces (Bach, Beethoven, Chopin, Mozart, Pachelbel).
2. **Music21 Preprocessing:** Quantized extraction of single notes, polyphonic chords, and durations.
3. **Deep Stacked LSTM Architecture:** Embedding -> Multi-Layer LSTM -> Dropout -> BatchNormalization -> Dense Softmax.
4. **Robust Training System:** Checkpoints, Early Stopping, Learning Rate Decay, and resume capability.
5. **Temperature-Based Sampling:** Controls diversity, musical coherence, and creative variation ($T \in [0.2, 1.2]$).
6. **MIDI & Audio Synthesis:** Generates standard `.mid` and playable `.wav` audio directly in your browser.


## 🖥️ Step 1: Check Runtime & GPU Acceleration
Google Colab offers free GPU acceleration. Make sure GPU is enabled via:
> **Runtime** ➔ **Change runtime type** ➔ **Hardware accelerator** ➔ **T4 GPU**.


In [ ]:
import sys
import tensorflow as tf

print(f"Python Version: {sys.version.split()[0]}")
print(f"TensorFlow Version: {tf.__version__}")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU Acceleration Active: {[gpu.name for gpu in gpus]}")
    !nvidia-smi
else:
    print("⚠️ No GPU detected. Running in CPU mode. (Tip: enable T4 GPU in Runtime settings for faster training)")


## 📦 Step 2: Install Dependencies & FluidSynth (Free Audio Tools)
We install `music21` for symbolic music representation, and `fluidsynth` for zero-cost audio rendering in Colab.


In [ ]:
!pip install --quiet music21 pretty_midi matplotlib
!apt-get install -y fluidsynth fluid-soundfont-gm > /dev/null 2>&1
print("✅ Dependencies installed successfully!")


## 📂 Step 3: Clone Repository or Setup Project Directory
If running directly on Google Colab, we clone your CodeAlpha repository or initialize the workspace.


In [ ]:
import os
import glob
from pathlib import Path

# If repository is cloned or files uploaded:
REPO_NAME = "CodeAlpha_MusicGenerationAI"
if not os.path.exists(REPO_NAME) and not os.path.exists("src"):
    !git clone https://github.com/YourUsername/CodeAlpha_MusicGenerationAI.git
    if os.path.exists(REPO_NAME):
        %cd CodeAlpha_MusicGenerationAI

# Create directories if running standalone
for d in ["data/raw_midi", "data/processed", "models", "output"]:
    os.makedirs(d, exist_ok=True)

print("Current Directory:", os.getcwd())


## 🎼 Step 4: Prepare & Inspect Curated Public-Domain MIDI Dataset
We use legally usable, public domain classical masterpieces (Bach, Beethoven, Chopin, Mozart, Pachelbel).


In [ ]:
import struct

def encode_vlq(value):
    buf = [value & 0x7F]
    value >>= 7
    while value > 0:
        buf.append((value & 0x7F) | 0x80)
        value >>= 7
    buf.reverse()
    return bytes(buf)

def write_smf(track_notes, output_path, ticks_per_beat=480, tempo_bpm=100):
    track_data = bytearray()
    tempo_us = int(60000000 / tempo_bpm)
    track_data.extend(encode_vlq(0))
    track_data.extend(bytes([0xFF, 0x51, 0x03, (tempo_us >> 16) & 0xFF, (tempo_us >> 8) & 0xFF, tempo_us & 0xFF]))
    track_data.extend(encode_vlq(0))
    track_data.extend(bytes([0xFF, 0x58, 0x04, 0x04, 0x02, 0x18, 0x08]))

    for item in track_notes:
        pitches, dur_beats, vel = item
        ticks = int(dur_beats * ticks_per_beat)
        if isinstance(pitches, (list, tuple)):
            for p in pitches:
                track_data.extend(encode_vlq(0))
                track_data.extend(bytes([0x90, p, vel]))
            for i, p in enumerate(pitches):
                delta = ticks if i == 0 else 0
                track_data.extend(encode_vlq(delta))
                track_data.extend(bytes([0x80, p, 0]))
        else:
            track_data.extend(encode_vlq(0))
            track_data.extend(bytes([0x90, pitches, vel]))
            track_data.extend(encode_vlq(ticks))
            track_data.extend(bytes([0x80, pitches, 0]))

    track_data.extend(encode_vlq(0))
    track_data.extend(bytes([0xFF, 0x2F, 0x00]))
    header = struct.pack(">4sIHHH", b"MThd", 6, 0, 1, ticks_per_beat)
    track_chunk = struct.pack(">4sI", b"MTrk", len(track_data)) + track_data

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, "wb") as f:
        f.write(header + track_chunk)

# Ensure curated files exist in data/raw_midi
raw_dir = "data/raw_midi"
if len(glob.glob(os.path.join(raw_dir, "*.mid"))) == 0:
    print("Writing curated public-domain classical pieces...")
    write_smf([(60,0.5,90),(62,0.5,90),(64,0.5,90),(65,0.5,90),(62,0.5,90),(64,0.5,90),(60,0.5,90),(67,1.0,95),((60,64,67),2.0,95)], f"{raw_dir}/bach_inv1.mid", tempo_bpm=100)
    write_smf([(64,1.0,90),(64,1.0,90),(65,1.0,90),(67,1.0,95),(67,1.0,95),(65,1.0,90),(64,1.0,90),(62,1.0,85),((60,64,67,72),3.0,100)], f"{raw_dir}/beethoven_ode.mid", tempo_bpm=120)
    write_smf([(71,3.0,80),(72,0.5,85),(71,0.5,85),(70,2.0,80),((52,59,64,67),4.0,70)], f"{raw_dir}/chopin_op28.mid", tempo_bpm=65)

midi_files = sorted(glob.glob(os.path.join(raw_dir, "*.mid")))
print(f"✅ Found {len(midi_files)} MIDI files in {raw_dir}:")
for f in midi_files:
    print(f"  - {os.path.basename(f)} ({os.path.getsize(f)} bytes)")


## 🎶 Step 5: Music21 Preprocessing Pipeline
We parse each MIDI file into **note, chord, and duration tokens** (e.g. `C4_0.5`, `C4.E4.G4_1.0`).
Then we construct sliding window sequences of length 32 for the LSTM to learn temporal transitions.


In [ ]:
import music21
import pickle
import json

def parse_midi(file_path):
    tokens = []
    try:
        midi = music21.converter.parse(file_path)
        parts = music21.instrument.partitionByInstrument(midi)
        elements = parts.parts[0].recurse() if parts else midi.flat.notesAndRests
        for el in elements:
            dur = max(0.25, round(round(float(el.duration.quarterLength) / 0.25) * 0.25, 2))
            if isinstance(el, music21.note.Note):
                tokens.append(f"{el.pitch}_{dur}")
            elif isinstance(el, music21.chord.Chord):
                chord_pitches = sorted([str(n.pitch) for n in el.notes])
                tokens.append(f"{'.'.join(chord_pitches)}_{dur}")
            elif isinstance(el, music21.note.Rest):
                tokens.append(f"rest_{dur}")
    except Exception as e:
        print(f"Notice parsing {file_path}: {e}")
    return tokens

# Extract all tokens
all_tokens = []
for f in midi_files:
    toks = parse_midi(f)
    print(f"Parsed {os.path.basename(f)}: {len(toks)} notes/chords")
    all_tokens.extend(toks)

print(f"\nTotal Tokens Extracted: {len(all_tokens)}")

# Build deterministic vocabulary
vocab = sorted(list(set(all_tokens)))
vocab_size = len(vocab)
token_to_idx = {tok: i for i, tok in enumerate(vocab)}
idx_to_token = {i: tok for i, tok in enumerate(vocab)}

print(f"Vocabulary Size: {vocab_size} distinct pitch-duration tokens")

# Sliding window sequence creation
SEQUENCE_LENGTH = 32
X, y = [], []
for i in range(0, len(all_tokens) - SEQUENCE_LENGTH):
    X.append([token_to_idx[t] for t in all_tokens[i:i+SEQUENCE_LENGTH]])
    y.append(token_to_idx[all_tokens[i+SEQUENCE_LENGTH]])

import numpy as np
X = np.array(X, dtype=np.int32)
y = np.array(y, dtype=np.int32)

print(f"Generated {len(X)} training sequences (Shape X: {X.shape}, Shape y: {y.shape})")


## 🧠 Step 6: Define Deep Stacked LSTM Neural Network
We construct a multi-layer LSTM model with:
- **Embedding Layer:** Projects discrete musical tokens into a continuous semantic harmonic space.
- **Stacked LSTMs:** 2 LSTM layers (256 units each) capturing local melody and long-term harmonic motifs.
- **Dropout & BatchNormalization:** Prevents overfitting and accelerates convergence.
- **Dense Softmax:** Computes probability distribution over the entire vocabulary.


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dropout, Dense, BatchNormalization, Input
from tensorflow.keras.optimizers import Adam

def build_model(vocab_size, seq_len=32, emb_dim=128):
    model = Sequential([
        Input(shape=(seq_len,)),
        Embedding(input_dim=vocab_size, output_dim=emb_dim, input_length=seq_len, name="token_embedding"),
        LSTM(256, return_sequences=True, name="lstm_layer_1"),
        Dropout(0.3, name="dropout_1"),
        BatchNormalization(name="batch_norm_1"),
        LSTM(256, return_sequences=False, name="lstm_layer_2"),
        Dropout(0.3, name="dropout_2"),
        BatchNormalization(name="batch_norm_2"),
        Dense(256, activation="relu", name="dense_hidden"),
        BatchNormalization(name="batch_norm_3"),
        Dropout(0.3, name="dropout_3"),
        Dense(vocab_size, activation="softmax", name="output_probabilities")
    ], name="CodeAlpha_Music_LSTM")

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

model = build_model(vocab_size=vocab_size, seq_len=SEQUENCE_LENGTH)
model.summary()


## 🚀 Step 7: Train the Model with Checkpoints & Resume Support
We train the model using:
- `ModelCheckpoint`: Automatically saves the best weights when validation loss improves.
- `EarlyStopping`: Protects against overfitting.
- `ReduceLROnPlateau`: Dynamically halves the learning rate when loss plateaus.


In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

checkpoint_path = "models/lstm_music_model_best.keras"
callbacks = [
    ModelCheckpoint(checkpoint_path, monitor="loss", save_best_only=True, verbose=1),
    EarlyStopping(monitor="loss", patience=8, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="loss", factor=0.5, patience=3, min_lr=1e-5, verbose=1)
]

# Set epochs (e.g. 30-50 on Colab GPU takes ~1-2 minutes)
EPOCHS = 35
BATCH_SIZE = 32

print("Starting Model Training...")
history = model.fit(
    X, y,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)
print("\n✅ Training Complete! Best weights saved to:", checkpoint_path)


## 📊 Step 8: Visualize Training Convergence


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="Training Loss", color="#1f77b4", lw=2)
plt.title("LSTM Loss Convergence")
plt.xlabel("Epoch")
plt.ylabel("Sparse Crossentropy Loss")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="Accuracy", color="#2ca02c", lw=2)
plt.title("Next-Token Prediction Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

plt.tight_layout()
plt.savefig("output/training_loss_curve.png", dpi=200)
plt.show()


## 🎹 Step 9: Temperature-Controlled AI Music Generation
Temperature scales the softmax distribution:
$$\text{scaled\_logits}_i = \frac{\ln(p_i)}{T}$$
- **$T = 0.3$:** Highly structured, conservative motif repetition.
- **$T = 0.8$:** **The Sweet Spot** — creative harmonic exploration while preserving musical coherence.
- **$T = 1.2$:** Freeform, experimental improvisation.


In [ ]:
import random

def sample_with_temperature(probabilities, temperature=0.8):
    if temperature <= 0.01:
        return int(np.argmax(probabilities))
    probs = np.asarray(probabilities).astype('float64')
    probs = np.clip(probs, 1e-10, 1.0)
    log_probs = np.log(probs) / temperature
    exp_preds = np.exp(log_probs)
    scaled_probs = exp_preds / np.sum(exp_preds)
    return int(np.argmax(np.random.multinomial(1, scaled_probs, 1)))

# Select a random seed sequence from training data
seed_idx = random.randint(0, len(X) - 1)
pattern = list(X[seed_idx])

# Hyperparameters for generation
GENERATION_LENGTH = 120   # Number of notes to generate
TEMPERATURE = 0.8         # Change this to experiment (e.g. 0.4, 0.8, 1.2)

generated_tokens = []
print(f"Generating {GENERATION_LENGTH} notes with Temperature = {TEMPERATURE}...")

for step in range(GENERATION_LENGTH):
    input_window = np.array([pattern[-SEQUENCE_LENGTH:]], dtype=np.int32)
    predictions = model.predict(input_window, verbose=0)[0]
    next_idx = sample_with_temperature(predictions, temperature=TEMPERATURE)
    next_token = idx_to_token[next_idx]
    generated_tokens.append(next_token)
    pattern.append(next_idx)

print("\nFirst 15 Generated Tokens:", generated_tokens[:15])


## 💾 Step 10: Convert Tokens to MIDI and Download


In [ ]:
def tokens_to_midi(tokens, output_file="output/ai_generated_music.mid", tempo_bpm=100):
    stream_part = music21.stream.Part()
    stream_part.append(music21.tempo.MetronomeMark(number=tempo_bpm))
    stream_part.append(music21.instrument.Piano())

    offset = 0.0
    for tok in tokens:
        parts = tok.rsplit("_", 1)
        pitch_str = parts[0]
        dur = float(parts[1]) if len(parts) > 1 else 0.5
        
        if pitch_str.lower() == "rest":
            r = music21.note.Rest()
            r.quarterLength = dur
            r.offset = offset
            stream_part.append(r)
        elif "." in pitch_str:
            chord_notes = [music21.note.Note(p) for p in pitch_str.split(".")]
            c = music21.chord.Chord(chord_notes)
            c.quarterLength = dur
            c.offset = offset
            stream_part.append(c)
        else:
            n = music21.note.Note(pitch_str)
            n.quarterLength = dur
            n.offset = offset
            stream_part.append(n)
        offset += dur

    score = music21.stream.Score([stream_part])
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    score.write("midi", fp=output_file)
    print(f"✅ Successfully wrote MIDI to: {output_file} ({os.path.getsize(output_file)} bytes)")
    return output_file

output_midi = tokens_to_midi(generated_tokens, "output/ai_generated_music.mid")

# Trigger browser download in Google Colab
try:
    from google.colab import files
    files.download(output_midi)
except Exception:
    print(f"MIDI ready at {output_midi}")


## 🔊 Step 11: Synthesize Audio to WAV & In-Browser Playback


In [ ]:
import math
import wave
from IPython.display import Audio, display

def synthesize_wav(tokens, output_wav="output/ai_generated_music.wav", tempo_bpm=100, sample_rate=44100):
    pitch_map = {'C': 0, 'C#': 1, 'D-': 1, 'D': 2, 'D#': 3, 'E-': 3, 'E': 4, 'F': 5, 'F#': 6, 'G-': 6, 'G': 7, 'G#': 8, 'A-': 8, 'A': 9, 'A#': 10, 'B-': 10, 'B': 11}
    def note_to_midi(name):
        name = name.strip()
        if len(name) >= 2 and name[-1].isdigit():
            octave = int(name[-1])
            letter = name[:-1].capitalize()
        else:
            octave = 4
            letter = name.capitalize()
        return (octave + 1) * 12 + pitch_map.get(letter, 0)

    sec_per_beat = 60.0 / tempo_bpm
    samples = []
    
    for tok in tokens:
        parts = tok.rsplit("_", 1)
        pitch_str = parts[0]
        dur = float(parts[1]) if len(parts) > 1 else 0.5
        dur_sec = dur * sec_per_beat
        n_samples = int(dur_sec * sample_rate)
        
        if pitch_str.lower() == "rest":
            samples.extend([0.0] * n_samples)
            continue
            
        pitches = [note_to_midi(p) for p in pitch_str.split(".")]
        for i in range(n_samples):
            t = i / sample_rate
            env = min(1.0, t / 0.01) * math.exp(-3.0 * t / max(0.1, dur_sec))
            val = sum(0.6*math.sin(2*math.pi*(440.0*2**((p-69)/12))*t) + 0.3*math.sin(4*math.pi*(440.0*2**((p-69)/12))*t) for p in pitches) / len(pitches)
            samples.append(val * env * 0.7)

    with wave.open(output_wav, "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sample_rate)
        raw = bytearray()
        for s in samples:
            raw.extend(struct.pack("<h", int(max(-1.0, min(1.0, s)) * 32767)))
        wf.writeframes(raw)
        
    print(f"✅ Synthesized WAV audio: {output_wav} ({len(samples)/sample_rate:.2f} seconds)")
    return output_wav

wav_file = synthesize_wav(generated_tokens, "output/ai_generated_music.wav")
display(Audio(wav_file, rate=44100, autoplay=False))


---
### 🎉 Congratulations!
You have successfully preprocessed MIDI music, trained a Deep Stacked LSTM model with Keras, sampled music with temperature control, and exported playable MIDI and WAV files.

**CodeAlpha Internship Submission Checklist:**
- [x] Curated public-domain MIDI dataset loaded
- [x] Music21 tokenization and sliding-window preprocessing
- [x] TensorFlow/Keras LSTM model compiled & trained
- [x] Checkpoint saved with resume support
- [x] Temperature-scaled music generation tested
- [x] Downloadable `.mid` generated
- [x] Playable `.wav` audio synthesized
